# Compare Dafne - to ground truth (extended metrics)

Note: this notebook was developed with assistance from Generative AI. The end cross schecks the results against a hand-coded earlier version

In [ ]:
# libraries
import os
import re
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import SimpleITK as sitk
from dissector.evaluation import binary_cross_entropy, boundary_iou_3d, inter_slice_dice

In [ ]:
!pwd
# above should be about ~something~/~someProjects~/dissector/eval_notebooks/myosegmenTUM

In [ ]:
# dilation distance used for boundary_iou_3d
BOUNDARY_DISTANCE = 1

## Right gracilis

In [ ]:
# unpack one npz to see muscle keys and find right gracilis
sample_npz = next(f for f in os.listdir("..") if f.endswith(".npz"))
data = np.load(os.path.join("..", sample_npz))

print(f"File: {sample_npz}\n")
for i, key in enumerate(data.files):
    marker = "  <-- RIGHT GRACILIS" if "gracilis" in key.lower() and "_r" in key.lower() else ""
    print(f"  [{i}] {key}{marker}")

In [ ]:
!pwd

In [ ]:
#find right gracilis

In [ ]:
results_r_gracilis = []
gt_base = os.path.join("..", "..", "myosegmenTUM")

for file in os.listdir(os.path.join("..")):
    if not file.endswith(".npz"):
        continue
    print(file)

    subject = file.split("_FATFRACTION")[0]
    stack_match = re.search(r"stack(\d+)", file)
    if not stack_match:
        print("  could not parse stack number, skipping")
        continue
    stack_num = stack_match.group(1)

    gt_name = os.path.join(gt_base, subject, "SegmentationMasks", f"combined_gt_stack{stack_num}.mha")
    print(gt_name)

    npz_data = np.load(os.path.join("..", file))
    pred_arr = npz_data["Gracilis_R"].astype(float)

    gt_image = sitk.ReadImage(gt_name)
    gt = sitk.Cast(gt_image == 5, sitk.sitkUInt8)
    gt_arr = sitk.GetArrayFromImage(gt).astype(float)

    pred_sitk = sitk.GetImageFromArray(pred_arr.astype(np.uint8))
    pred_sitk.CopyInformation(gt_image)
    pred = sitk.Cast(pred_sitk, sitk.sitkUInt8)

    dice_filter = sitk.LabelOverlapMeasuresImageFilter()
    dice_filter.Execute(gt, pred)
    right_grac_dice_lower = dice_filter.GetDiceCoefficient()
    right_JaccardCoeffi = dice_filter.GetJaccardCoefficient()
    right_VolumeSimilar = dice_filter.GetVolumeSimilarity()
    right_FalseNegative = dice_filter.GetFalseNegativeError()
    right_FalsePositive = dice_filter.GetFalsePositiveError()

    if gt_arr.sum() > 0 and pred_arr.sum() > 0:
        hd_filter = sitk.HausdorffDistanceImageFilter()
        hd_filter.Execute(gt, pred)
        right_grac_hd_lower = hd_filter.GetHausdorffDistance()
    else:
        print(f"  empty mask (gt={int(gt_arr.sum())} pred={int(pred_arr.sum())}), HD set to NaN")
        right_grac_hd_lower = np.nan

    bce = binary_cross_entropy(gt_arr, pred_arr)
    biou = boundary_iou_3d(BOUNDARY_DISTANCE, gt_arr, pred_arr)
    isd_pred = inter_slice_dice(pred_arr)
    isd_gt   = inter_slice_dice(gt_arr)

    results_r_gracilis.append({
        "image": gt_name,
        "pred_label": file,
        "R_gracilis_lower_dice:": right_grac_dice_lower,
        "R_gracilis_lower_Hausdorff:": right_grac_hd_lower,
        "R_gracilis_jaccard": right_JaccardCoeffi,
        "R_gracilis_volume_similarity": right_VolumeSimilar,
        "R_gracilis_falseNegative": right_FalseNegative,
        "R_gracilis_falsePostivie": right_FalsePositive,
        "R_gracilis_binary_cross_entropy": bce,
        "R_gracilis_boundary_iou_3d": biou,
        "R_gracilis_inter_slice_dice_pred": isd_pred,
        "R_gracilis_inter_slice_dice_gt": isd_gt,
    })

df_r_gracilis = pd.DataFrame(results_r_gracilis)
df_r_gracilis.to_csv("../results/df_r_gracilis_dafne.csv")
df_r_gracilis

## Left gracilis

In [ ]:
results_l_gracilis = []
gt_base = os.path.join("..", "..", "myosegmenTUM")

for file in os.listdir(os.path.join("..")):
    if not file.endswith(".npz"):
        continue
    print(file)

    subject = file.split("_FATFRACTION")[0]
    stack_match = re.search(r"stack(\d+)", file)
    if not stack_match:
        print("  could not parse stack number, skipping")
        continue
    stack_num = stack_match.group(1)

    gt_name = os.path.join(gt_base, subject, "SegmentationMasks", f"combined_gt_stack{stack_num}.mha")
    print(gt_name)

    npz_data = np.load(os.path.join("..", file))
    pred_arr = npz_data["Gracilis_L"].astype(float)

    gt_image = sitk.ReadImage(gt_name)
    gt = sitk.Cast(gt_image == 1, sitk.sitkUInt8)
    gt_arr = sitk.GetArrayFromImage(gt).astype(float)

    pred_sitk = sitk.GetImageFromArray(pred_arr.astype(np.uint8))
    pred_sitk.CopyInformation(gt_image)
    pred = sitk.Cast(pred_sitk, sitk.sitkUInt8)

    dice_filter = sitk.LabelOverlapMeasuresImageFilter()
    dice_filter.Execute(gt, pred)
    left_grac_dice_lower = dice_filter.GetDiceCoefficient()
    left_JaccardCoeffi = dice_filter.GetJaccardCoefficient()
    left_VolumeSimilar = dice_filter.GetVolumeSimilarity()
    left_FalseNegative = dice_filter.GetFalseNegativeError()
    left_FalsePositive = dice_filter.GetFalsePositiveError()

    if gt_arr.sum() > 0 and pred_arr.sum() > 0:
        hd_filter = sitk.HausdorffDistanceImageFilter()
        hd_filter.Execute(gt, pred)
        left_grac_hd_lower = hd_filter.GetHausdorffDistance()
    else:
        print(f"  empty mask (gt={int(gt_arr.sum())} pred={int(pred_arr.sum())}), HD set to NaN")
        left_grac_hd_lower = np.nan

    bce = binary_cross_entropy(gt_arr, pred_arr)
    biou = boundary_iou_3d(BOUNDARY_DISTANCE, gt_arr, pred_arr)
    isd_pred = inter_slice_dice(pred_arr)
    isd_gt   = inter_slice_dice(gt_arr)

    results_l_gracilis.append({
        "image": gt_name,
        "pred_label": file,
        "L_gracilis_lower_dice:": left_grac_dice_lower,
        "L_gracilis_lower_Hausdorff:": left_grac_hd_lower,
        "L_gracilis_jaccard": left_JaccardCoeffi,
        "L_gracilis_volume_similarity": left_VolumeSimilar,
        "L_gracilis_falseNegative": left_FalseNegative,
        "L_gracilis_falsePostivie": left_FalsePositive,
        "L_gracilis_binary_cross_entropy": bce,
        "L_gracilis_boundary_iou_3d": biou,
        "L_gracilis_inter_slice_dice_pred": isd_pred,
        "L_gracilis_inter_slice_dice_gt": isd_gt,
    })

df_l_gracilis = pd.DataFrame(results_l_gracilis)
df_l_gracilis.to_csv("../results/df_l_gracilis_dafne.csv")
df_l_gracilis

## Right sartorius

In [ ]:
results_r_sart = []
gt_base = os.path.join("..", "..", "myosegmenTUM")

for file in os.listdir(os.path.join("..")):
    if not file.endswith(".npz"):
        continue
    print(file)

    subject = file.split("_FATFRACTION")[0]
    stack_match = re.search(r"stack(\d+)", file)
    if not stack_match:
        print("  could not parse stack number, skipping")
        continue
    stack_num = stack_match.group(1)

    gt_name = os.path.join(gt_base, subject, "SegmentationMasks", f"combined_gt_stack{stack_num}.mha")
    print(gt_name)

    npz_data = np.load(os.path.join("..", file))
    pred_arr = npz_data["Sartorius_R"].astype(float)

    gt_image = sitk.ReadImage(gt_name)
    gt = sitk.Cast(gt_image == 8, sitk.sitkUInt8)
    gt_arr = sitk.GetArrayFromImage(gt).astype(float)

    pred_sitk = sitk.GetImageFromArray(pred_arr.astype(np.uint8))
    pred_sitk.CopyInformation(gt_image)
    pred = sitk.Cast(pred_sitk, sitk.sitkUInt8)

    dice_filter = sitk.LabelOverlapMeasuresImageFilter()
    dice_filter.Execute(gt, pred)
    right_sart_dice_lower = dice_filter.GetDiceCoefficient()
    right_JaccardCoeffi = dice_filter.GetJaccardCoefficient()
    right_VolumeSimilar = dice_filter.GetVolumeSimilarity()
    right_FalseNegative = dice_filter.GetFalseNegativeError()
    right_FalsePositive = dice_filter.GetFalsePositiveError()

    if gt_arr.sum() > 0 and pred_arr.sum() > 0:
        hd_filter = sitk.HausdorffDistanceImageFilter()
        hd_filter.Execute(gt, pred)
        right_sart_hd_lower = hd_filter.GetHausdorffDistance()
    else:
        print(f"  empty mask (gt={int(gt_arr.sum())} pred={int(pred_arr.sum())}), HD set to NaN")
        right_sart_hd_lower = np.nan

    bce = binary_cross_entropy(gt_arr, pred_arr)
    biou = boundary_iou_3d(BOUNDARY_DISTANCE, gt_arr, pred_arr)
    isd_pred = inter_slice_dice(pred_arr)
    isd_gt   = inter_slice_dice(gt_arr)

    results_r_sart.append({
        "image": gt_name,
        "pred_label": file,
        "R_sart_lower_dice:": right_sart_dice_lower,
        "R_sart_lower_Hausdorff:": right_sart_hd_lower,
        "R_sart_jaccard": right_JaccardCoeffi,
        "R_sart_volume_similarity": right_VolumeSimilar,
        "R_sart_falseNegative": right_FalseNegative,
        "R_sart_falsePostivie": right_FalsePositive,
        "R_sart_binary_cross_entropy": bce,
        "R_sart_boundary_iou_3d": biou,
        "R_sart_inter_slice_dice_pred": isd_pred,
        "R_sart_inter_slice_dice_gt": isd_gt,
    })

df_r_sart = pd.DataFrame(results_r_sart)
df_r_sart.to_csv("../results/df_r_sart_dafne.csv")
df_r_sart

## Left sartorius

In [ ]:
results_l_sart = []
gt_base = os.path.join("..", "..", "myosegmenTUM")

for file in os.listdir(os.path.join("..")):
    if not file.endswith(".npz"):
        continue
    print(file)

    subject = file.split("_FATFRACTION")[0]
    stack_match = re.search(r"stack(\d+)", file)
    if not stack_match:
        print("  could not parse stack number, skipping")
        continue
    stack_num = stack_match.group(1)

    gt_name = os.path.join(gt_base, subject, "SegmentationMasks", f"combined_gt_stack{stack_num}.mha")
    print(gt_name)

    npz_data = np.load(os.path.join("..", file))
    pred_arr = npz_data["Sartorius_L"].astype(float)

    gt_image = sitk.ReadImage(gt_name)
    gt = sitk.Cast(gt_image == 4, sitk.sitkUInt8)
    gt_arr = sitk.GetArrayFromImage(gt).astype(float)

    pred_sitk = sitk.GetImageFromArray(pred_arr.astype(np.uint8))
    pred_sitk.CopyInformation(gt_image)
    pred = sitk.Cast(pred_sitk, sitk.sitkUInt8)

    dice_filter = sitk.LabelOverlapMeasuresImageFilter()
    dice_filter.Execute(gt, pred)
    left_sart_dice_lower = dice_filter.GetDiceCoefficient()
    left_JaccardCoeffi = dice_filter.GetJaccardCoefficient()
    left_VolumeSimilar = dice_filter.GetVolumeSimilarity()
    left_FalseNegative = dice_filter.GetFalseNegativeError()
    left_FalsePositive = dice_filter.GetFalsePositiveError()

    if gt_arr.sum() > 0 and pred_arr.sum() > 0:
        hd_filter = sitk.HausdorffDistanceImageFilter()
        hd_filter.Execute(gt, pred)
        left_sart_hd_lower = hd_filter.GetHausdorffDistance()
    else:
        print(f"  empty mask (gt={int(gt_arr.sum())} pred={int(pred_arr.sum())}), HD set to NaN")
        left_sart_hd_lower = np.nan

    bce = binary_cross_entropy(gt_arr, pred_arr)
    biou = boundary_iou_3d(BOUNDARY_DISTANCE, gt_arr, pred_arr)
    isd_pred = inter_slice_dice(pred_arr)
    isd_gt   = inter_slice_dice(gt_arr)

    results_l_sart.append({
        "image": gt_name,
        "pred_label": file,
        "L_sart_lower_dice:": left_sart_dice_lower,
        "L_sart_lower_Hausdorff:": left_sart_hd_lower,
        "L_sart_jaccard": left_JaccardCoeffi,
        "L_sart_volume_similarity": left_VolumeSimilar,
        "L_sart_falseNegative": left_FalseNegative,
        "L_sart_falsePostivie": left_FalsePositive,
        "L_sart_binary_cross_entropy": bce,
        "L_sart_boundary_iou_3d": biou,
        "L_sart_inter_slice_dice_pred": isd_pred,
        "L_sart_inter_slice_dice_gt": isd_gt,
    })

df_l_sart = pd.DataFrame(results_l_sart)
df_l_sart.to_csv("../results/df_l_sart_dafne.csv")
df_l_sart